Following the data and simulation files processing, here we:

- Merge processed files by production conditions.
- Tag global events by type of particle (electron or alpha-like) and detector region.
- Store final HDF5 merged file for subsequent analysis.

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis/')

from libs import crudo

from datetime import datetime
import glob
import os
import pandas as pd
import numpy as np

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Configuration

In [2]:
# ----- Notebook Details
TYPE = 'mc'                     # Options: 'data', 'mc'
FILE_TAG = 'radiogenics_lpr'    # Options: 'radiogenics_hpr', 'radiogenics_lpr', 'bb2nu_hpr', 'bb2nu_lpr', 'bb0nu_hpr', 'bb0nu_lpr', 'p2_zemrude', 'p2_icaros', 'p2_final'

# Data
DATA_PERIOD = 2                 # Options: 1, 2
DETECTOR_CONDITION = 'castle_closed_RAS'       # Options: None, 'castle_open', 'castle_closed', 'castle_closed_RAS', 'castle_pclosed', 'castle_pclosed_RAS'

# MC
DATE = datetime.now().strftime('%d%m%Y')    # Options: today or some day (e.g '02122025')

In [18]:
# ------------------------------
# DIRECTORIES, PATHS & FILENAMES
# ------------------------------
PROC_DATA_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/runs/'
PROC_MC_DIR   = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/'
OUTPUT_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/'

RUNS_INFO_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Backgrounds/utilities/runs_information.csv')

SUMMARY_FILENAME = 'summary_' + FILE_TAG +'_processed.csv'
SUMMARY_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Backgrounds/txt/summaries/', SUMMARY_FILENAME)

# -----------------------------
# ALPHA/ELECTRON DISCRIMINATION
# -----------------------------
SIZE_THRESHOLD = 2e3          # in [# of hits]
S1_ENERGY_THRESHOLD = 900     # in [PE]

# ----------------
# DETECTOR REGIONS
# ----------------
# Geometric boundaries for event classification.
Z_LOW = 40          # in [mm]
Z_UP  = 1147        # in [mm]
R_UP  = 451.65      # in [mm]

# -------------
# FINAL COLUMNS
# -------------
INFO_COLS = ['event', 'global_event', 'time']
MC_COLS   = ['isotope', 'volume', 'double_e', 'region']
DATA_COLS = ['run_number', 'particle', 'region']
EVT_COLS = ['nS1', 'nS2', 'n_cluster', 'old_n_hits', 'n_hits', 'E_evt_mev']
POS_COLS = ['X_bary', 'Y_bary', 'Z_bary', 'X_min', 'X_max', 'Y_min', 'Y_max', 'Z_min', 'Z_max', 'R_max']
S1_PULSE_COLS = ['S1e', 'S1e_corr', 'S1w', 'S1h', 'S1t']
S2_PULSE_COLS = ['S2e', 'S2w', 'S2h', 'S2t', 'S2q']

if TYPE == 'mc':
    FINAL_COLS = INFO_COLS + MC_COLS + EVT_COLS + POS_COLS + S1_PULSE_COLS + S2_PULSE_COLS
elif TYPE == 'data':
    FINAL_COLS = INFO_COLS + DATA_COLS + EVT_COLS + POS_COLS + S1_PULSE_COLS + S2_PULSE_COLS  

### Runs & Summary Information

In [4]:
# Runs information
RUNS_INFO_DF = pd.read_csv(RUNS_INFO_PATH)
RUNS_INFO_DF.columns = RUNS_INFO_DF.columns.str.strip()
RUNS_INFO_DF

,run_number,duration,OK,LOST,period,condition
0,15062,84783,69564,1339,1,castle_open
1,15063,79120,65052,1241,1,castle_open
2,15076,69316,56775,1080,1,castle_open
3,15288,87256,30201,8397,1,castle_pclosed_RAS
4,15289,82152,28180,7884,1,castle_pclosed_RAS
...,...,...,...,...,...,...
106,15733,86919,30475,10027,2,castle_closed_RAS
107,15734,85790,29837,9598,2,castle_closed_RAS
108,15735,87451,30547,9958,2,castle_closed_RAS
109,15736,93376,32622,10506,2,castle_closed_RAS


In [5]:
# Summary of the processed runs
SUMMARY_DF = pd.read_csv(SUMMARY_PATH)
SUMMARY_DF.drop(columns=['Unnamed: 0'], inplace=True)
SUMMARY_DF.columns = SUMMARY_DF.columns.str.strip()
if TYPE == 'data':  SUMMARY_DF.sort_values(by='Run_ID', inplace=True)
SUMMARY_DF

,Isotope,Generated,Interacting,Saved,Reconstructed,Strong_S2,S1_Cut,Clean_Events
0,Bi214,2919330850,7462907,1375595,1217460,1217460,1214073,1207670
1,Co60,596157422,8199419,1301557,1290512,1290512,1286798,1286737
2,K40,5090350250,3315521,676953,395446,395446,394324,394160
3,Tl208,723337199,3429146,511413,480340,480340,479004,478817


# Merge by Production Condition

- For data, the production are differenciated by _data period_ and _detector condition_.
- For MC, by type of simulation.

### Data

In [6]:
# Select runs to use according to the notebook configuration
if DATA_PERIOD is not None:
    runs_to_analyze = RUNS_INFO_DF.loc[RUNS_INFO_DF['period'] == DATA_PERIOD, 'run_number'].values
    if DETECTOR_CONDITION is not None:
        runs_to_analyze = RUNS_INFO_DF.loc[(RUNS_INFO_DF['period'] == DATA_PERIOD) & (RUNS_INFO_DF['condition'] == DETECTOR_CONDITION), 'run_number'].values

# Selection
print(f"\nSelected {len(runs_to_analyze)} runs for merge:")
print(runs_to_analyze)


Selected 56 runs for merge:
[15625 15626 15627 15632 15633 15634 15635 15636 15637 15639 15640 15642
 15643 15644 15645 15647 15648 15649 15650 15655 15656 15657 15658 15659
 15669 15670 15671 15672 15673 15675 15676 15681 15682 15687 15688 15689
 15693 15694 15695 15696 15697 15698 15699 15700 15701 15709 15724 15729
 15730 15731 15732 15733 15734 15735 15736 15737]


In [7]:
total_corr_time = 0
# total_ok = 0
# total_lost = 0
total_processed_events = 0
all_processed_df = []

for run_id in runs_to_analyze:

    print(f"--- Merging Run {run_id} ---")
    if run_id not in SUMMARY_DF['Run_ID'].values:
        print(f"  → Run {run_id} not found in summary file. Skipping...")
        continue

    # --- Run Information --- #
    # Extract run information from the summary dataframe
    run_duration = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Duration'].values[0]
    run_OK   = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'OK'].values[0]
    run_LOST = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'LOST'].values[0]
    # Calculate DAQ efficiency and corrected time
    DAQe_CV, DAQe_error = crudo.ff.efficiency(run_OK, run_LOST)
    run_corr_time    = run_duration * DAQe_CV
    total_corr_time += run_corr_time
    # Accumulate processed events
    processed_events = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Clean_Events'].values[0]
    total_processed_events += processed_events

    # Add run_number to dataframe for the global_event_id
    run_file = os.path.join(PROC_DATA_DIR, f'processed_run_{run_id}_{VERSION_TAG}_events.h5')
    run_df = pd.read_hdf(run_file, key='Events')
    run_df['run_number'] = run_id
    all_processed_df.append(run_df)

# --- Print Summary --- #
print(f"\nFor period {DATA_PERIOD} with condition '{DETECTOR_CONDITION}':\n  Corrected Time = {total_corr_time:.4f} s")
# Concatenate all dataframes
MERGED_DF = pd.concat(all_processed_df, ignore_index=True)
print(f"Dataframe merged successfully:\n  Total processed events: {total_processed_events}")

--- Merging Run 15625 ---


KeyError: 'Run_ID'

### MC

In [8]:
mc_files_to_merge = sorted(glob.glob(os.path.join(PROC_MC_DIR, f"*{TYPE}*{DATE}*.h5")))
mc_files_to_merge

['/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/processed_mc_radiogenics_lpr_Bi214_07052026.h5',
 '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/processed_mc_radiogenics_lpr_Co60_07052026.h5',
 '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/processed_mc_radiogenics_lpr_K40_07052026.h5',
 '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/processed_mc_radiogenics_lpr_Tl208_07052026.h5']

In [9]:
all_processed_df = []

for file in mc_files_to_merge:
    print(f"--- Merging File: {os.path.basename(file)} ---")
    dataframe = pd.read_hdf(file, key='Events')
    all_processed_df.append(dataframe)

MERGED_DF = pd.concat(all_processed_df, ignore_index=True)

--- Merging File: processed_mc_radiogenics_lpr_Bi214_07052026.h5 ---
--- Merging File: processed_mc_radiogenics_lpr_Co60_07052026.h5 ---
--- Merging File: processed_mc_radiogenics_lpr_K40_07052026.h5 ---
--- Merging File: processed_mc_radiogenics_lpr_Tl208_07052026.h5 ---


,event,nS1,nS2,isotope,volume,double_e,old_n_hits,time,S1e,S1e_corr,...,X_bary,Y_bary,Z_bary,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max
0,0,1,2,Bi214,ANODE_RING,False,146,1.253990e+06,154.0,298.819863,...,-344.062292,-165.221148,33.491546,-404.875,-297.025,-232.625,-78.125,15.618709,44.207833,438.390558
1,0,1,2,Bi214,ANODE_RING,False,146,1.253990e+06,154.0,294.656683,...,-344.062292,-165.221148,33.491546,-404.875,-297.025,-232.625,-78.125,15.618709,44.207833,438.390558
2,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,413.197699,...,-126.622985,-96.187692,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941
3,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,409.465023,...,-126.622985,-96.187692,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941
4,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,219.883025,...,-126.622985,-96.187692,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4681188,278236188,1,1,Tl208,VESSEL,False,334,5.564724e+14,289.0,311.507711,...,154.670528,-15.947933,1001.394242,88.725,212.125,-94.175,44.775,985.971653,1020.127635,221.304024
4681189,278236189,1,1,Tl208,VESSEL,True,1159,5.564724e+14,941.0,989.822117,...,-214.016929,-288.520026,1109.927903,-312.575,-96.875,-371.075,-185.975,1044.224134,1196.774284,453.852373
4681190,278236190,1,2,Tl208,VESSEL,False,1148,5.564724e+14,1007.0,1510.253744,...,-211.773577,-170.616595,761.917078,-420.425,458.925,-279.275,-0.875,367.558851,951.444289,495.175581
4681191,278236190,1,2,Tl208,VESSEL,False,1148,5.564724e+14,1007.0,1119.689107,...,-211.773577,-170.616595,761.917078,-420.425,458.925,-279.275,-0.875,367.558851,951.444289,495.175581


### Compute Global Event ID

In [10]:
if TYPE == 'data': COMP_COLS = ['event', 'run_number']
elif TYPE == 'mc': COMP_COLS = ['event', 'isotope', 'volume']

# An original event is defined as a row in dataframe where at least one of the columns 
# ('event', 'run_number') differs from the corresponding row below it (using `shift`).
event_OG = (MERGED_DF[COMP_COLS] != MERGED_DF[COMP_COLS].shift())

# If any column in event_OG is True, it means the row corresponds to the start of a new original event block.
new_event_block = event_OG.any(axis=1)

# Use `cumsum()` on the boolean mask to create a unique identifier for each contiguous block of hits 
# that belong to the same original event.
unique_block_id = new_event_block.cumsum()

# Assign a unique global event ID to each block of original events.
# The `factorize` function generates a unique integer code for each unique block ID.
MERGED_DF['global_event'] = pd.factorize(unique_block_id)[0]
print(f"{MERGED_DF['global_event'].nunique()} unique global events identified.")

3367384 unique global events identified.


In [11]:
MERGED_DF

,event,nS1,nS2,isotope,volume,double_e,old_n_hits,time,S1e,S1e_corr,...,Y_bary,Z_bary,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max,global_event
0,0,1,2,Bi214,ANODE_RING,False,146,1.253990e+06,154.0,298.819863,...,-165.221148,33.491546,-404.875,-297.025,-232.625,-78.125,15.618709,44.207833,438.390558,0
1,0,1,2,Bi214,ANODE_RING,False,146,1.253990e+06,154.0,294.656683,...,-165.221148,33.491546,-404.875,-297.025,-232.625,-78.125,15.618709,44.207833,438.390558,0
2,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,413.197699,...,-96.187692,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1
3,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,409.465023,...,-96.187692,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1
4,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,219.883025,...,-96.187692,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4681188,278236188,1,1,Tl208,VESSEL,False,334,5.564724e+14,289.0,311.507711,...,-15.947933,1001.394242,88.725,212.125,-94.175,44.775,985.971653,1020.127635,221.304024,3367380
4681189,278236189,1,1,Tl208,VESSEL,True,1159,5.564724e+14,941.0,989.822117,...,-288.520026,1109.927903,-312.575,-96.875,-371.075,-185.975,1044.224134,1196.774284,453.852373,3367381
4681190,278236190,1,2,Tl208,VESSEL,False,1148,5.564724e+14,1007.0,1510.253744,...,-170.616595,761.917078,-420.425,458.925,-279.275,-0.875,367.558851,951.444289,495.175581,3367382
4681191,278236190,1,2,Tl208,VESSEL,False,1148,5.564724e+14,1007.0,1119.689107,...,-170.616595,761.917078,-420.425,458.925,-279.275,-0.875,367.558851,951.444289,495.175581,3367382


# Tagging Events

### By Particle

In [13]:
particle_tagged_MERGED_DF = crudo.dm.tag_particles( MERGED_DF
                                                  , size_threshold=SIZE_THRESHOLD
                                                  , s1_energy_threshold=S1_ENERGY_THRESHOLD
                                                  , event_column='global_event' )
particle_tagged_MERGED_DF

,event,nS1,nS2,isotope,volume,double_e,old_n_hits,time,S1e,S1e_corr,...,Z_bary,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max,global_event,particle
0,0,1,2,Bi214,ANODE_RING,False,146,1.253990e+06,154.0,298.819863,...,33.491546,-404.875,-297.025,-232.625,-78.125,15.618709,44.207833,438.390558,0,electron
1,0,1,2,Bi214,ANODE_RING,False,146,1.253990e+06,154.0,294.656683,...,33.491546,-404.875,-297.025,-232.625,-78.125,15.618709,44.207833,438.390558,0,electron
2,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,413.197699,...,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1,electron
3,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,409.465023,...,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1,electron
4,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,219.883025,...,786.343249,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1,electron
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4681188,278236188,1,1,Tl208,VESSEL,False,334,5.564724e+14,289.0,311.507711,...,1001.394242,88.725,212.125,-94.175,44.775,985.971653,1020.127635,221.304024,3367380,electron
4681189,278236189,1,1,Tl208,VESSEL,True,1159,5.564724e+14,941.0,989.822117,...,1109.927903,-312.575,-96.875,-371.075,-185.975,1044.224134,1196.774284,453.852373,3367381,alpha
4681190,278236190,1,2,Tl208,VESSEL,False,1148,5.564724e+14,1007.0,1510.253744,...,761.917078,-420.425,458.925,-279.275,-0.875,367.558851,951.444289,495.175581,3367382,alpha
4681191,278236190,1,2,Tl208,VESSEL,False,1148,5.564724e+14,1007.0,1119.689107,...,761.917078,-420.425,458.925,-279.275,-0.875,367.558851,951.444289,495.175581,3367382,alpha


### By Detector Region

In [14]:
region_tagged_MERGED_DF = crudo.dm.tag_event_by_detector_region( particle_tagged_MERGED_DF if TYPE == 'data' else MERGED_DF
                                                               , z_cut_low=Z_LOW
                                                               , z_cut_high=Z_UP
                                                               , r_cut_high=R_UP
                                                               , event_column='global_event' )
region_tagged_MERGED_DF

,event,nS1,nS2,isotope,volume,double_e,old_n_hits,time,S1e,S1e_corr,...,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max,global_event,particle,region
0,0,1,2,Bi214,ANODE_RING,False,146,1.253990e+06,154.0,298.819863,...,-404.875,-297.025,-232.625,-78.125,15.618709,44.207833,438.390558,0,electron,anode
1,0,1,2,Bi214,ANODE_RING,False,146,1.253990e+06,154.0,294.656683,...,-404.875,-297.025,-232.625,-78.125,15.618709,44.207833,438.390558,0,electron,anode
2,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,413.197699,...,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1,electron,anode
3,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,409.465023,...,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1,electron,anode
4,1,1,3,Bi214,ANODE_RING,False,245,2.387403e+06,214.0,219.883025,...,-250.375,88.725,-417.725,106.975,20.197641,1122.937601,421.680941,1,electron,anode
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4681188,278236188,1,1,Tl208,VESSEL,False,334,5.564724e+14,289.0,311.507711,...,88.725,212.125,-94.175,44.775,985.971653,1020.127635,221.304024,3367380,electron,fiducial
4681189,278236189,1,1,Tl208,VESSEL,True,1159,5.564724e+14,941.0,989.822117,...,-312.575,-96.875,-371.075,-185.975,1044.224134,1196.774284,453.852373,3367381,alpha,cathode
4681190,278236190,1,2,Tl208,VESSEL,False,1148,5.564724e+14,1007.0,1510.253744,...,-420.425,458.925,-279.275,-0.875,367.558851,951.444289,495.175581,3367382,alpha,tube
4681191,278236190,1,2,Tl208,VESSEL,False,1148,5.564724e+14,1007.0,1119.689107,...,-420.425,458.925,-279.275,-0.875,367.558851,951.444289,495.175581,3367382,alpha,tube


# Output

In [15]:
FINAL_DF = region_tagged_MERGED_DF[FINAL_COLS].copy()
FINAL_DF

,event,global_event,time,isotope,volume,double_e,region,nS1,nS2,n_cluster,...,S1e,S1e_corr,S1w,S1h,S1t,S2e,S2w,S2h,S2t,S2q
0,0,0,1.253990e+06,Bi214,ANODE_RING,False,anode,1,2,2,...,154.0,298.819863,500.0,29.0,10000.0,10881.0,6.900,4112.0,2.849166e+04,242.840591
1,0,0,1.253990e+06,Bi214,ANODE_RING,False,anode,1,2,2,...,154.0,294.656683,500.0,29.0,10000.0,142305.0,27.925,13222.0,4.849697e+04,4456.753906
2,1,1,2.387403e+06,Bi214,ANODE_RING,False,anode,1,3,3,...,214.0,413.197699,475.0,44.0,10000.0,37041.0,10.500,10077.0,3.550097e+04,1095.826050
3,1,1,2.387403e+06,Bi214,ANODE_RING,False,anode,1,3,3,...,214.0,409.465023,475.0,44.0,10000.0,11601.0,9.550,3318.0,4.847230e+04,304.000000
4,1,1,2.387403e+06,Bi214,ANODE_RING,False,anode,1,3,3,...,214.0,219.883025,475.0,44.0,10000.0,124104.0,87.900,3447.0,1.286492e+06,3416.463867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4681188,278236188,3367380,5.564724e+14,Tl208,VESSEL,False,fiducial,1,1,1,...,289.0,311.507711,550.0,64.0,10000.0,216042.0,61.400,9142.0,1.161488e+06,6508.000000
4681189,278236189,3367381,5.564724e+14,Tl208,VESSEL,True,cathode,1,1,1,...,941.0,989.822117,675.0,188.0,10000.0,636933.0,202.375,7319.0,1.224486e+06,23607.130859
4681190,278236190,3367382,5.564724e+14,Tl208,VESSEL,False,tube,1,2,2,...,1007.0,1510.253744,650.0,201.0,10000.0,86171.0,39.150,6457.0,4.444916e+05,2684.753662
4681191,278236190,3367382,5.564724e+14,Tl208,VESSEL,False,tube,1,2,2,...,1007.0,1119.689107,650.0,201.0,10000.0,615198.0,387.400,8205.0,1.083491e+06,28733.042969


In [19]:
# H5 output filename
merged_filename = 'merged_tagged_'
if TYPE == 'data': merged_filename += 'runs_' + DETECTOR_CONDITION + '_'
elif TYPE == 'mc': merged_filename += 'mc_'
merged_filename += FILE_TAG + '.h5'
    
merged_path = os.path.join(OUTPUT_DIR, merged_filename)
print(f"\nSaving merged dataframe to: {merged_path}")


Saving merged dataframe to: /lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/merged_tagged_mc_radiogenics_lpr.h5


In [20]:
FINAL_DF.to_hdf(merged_path, key='Events', mode='w', format='table')
print('Done!')

Done!
